# Отдельный эксперимент: Llama → структурированные признаки → LightGBM

# LLM-FE на H&M: Llama‑3.1‑8B‑Instruct, Kaggle T4

Llama выбирает операции, поля, окна и группировки в JSON; host компилирует их в SQL и строит признаки; LightGBM измеряет их качество и направляет
поиск в трёх популяциях. LightGBM используется на всех этапах: контроль, search,
confirmation и итоговая оценка. Ручной LightGBM и GNN→LightGBM сравниваются по сохранённым предсказаниям
на **тех же пользователях и кандидатах**. Это адаптация исследования, не его точное воспроизведение.

**Как запустить:**
1. В Kaggle импортируйте этот `.ipynb` через **File → Import Notebook**.
2. Выберите GPU **T4** (достаточно одной; при T4×2 используется GPU0), включите Internet.
3. Создайте Kaggle Dataset из `hm_llmfe_kaggle_fulltrain_v2.zip` и добавьте его через Input.
   Поддерживаются и исходный ZIP, и автоматически распакованные Kaggle файлы.
4. В конфиге выберите `MODEL_SOURCE = 'unsloth'` (по умолчанию, без токена) или `'hf'`.
   Для `'hf'` получите доступ к [официальной Llama](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)
   и добавьте `HF_TOKEN` в **Add-ons → Secrets**. Оба источника размещены на Hugging Face.
5. Выполните **Run All**. Модель скачивается автоматически; веса не входят в архив данных.
6. Скачайте итоговый `*_results.zip` из Output. В нём программы, модели деревьев, метрики и графики.

В ZIP данных находятся **те же803377 обучающих пар /50000 user-time строк /52 снимка**, что
в локальных сравнениях. Новое отрицательное семплирование не выполняется.
Ограничена только оценка: search256, confirmation512, отдельная val1024, test2048 пользователей.
Поисковые/confirmation/итоговые val строки не пересекаются. Кандидаты ID-GNN неизменны;
позитивные test-товары в пул дополнительно не добавляются.

Это новый ограниченный язык признаков, а не свободный SQL-интерфейс Qwen. Генерация Llama остаётся обычным sampling; маска токенов/grammar decoding не используется. Структура проверяется после генерации, ошибки отправляются на исправление.


## Условия и отличия от статьи

| Параметр | Этот ноутбук |
|---|---|
| LLM | Llama-3.1-8B-Instruct: официальный источник Meta (`hf`) или публикация Unsloth (`unsloth`) |
| Размещение на T4 | NF4 4-bit, вычисления float16; квантизация — наше отличие |
| Генерация | temperature0.8,3 новых предложения на один prompt; цель20 различных оценённых наборов |
| Бюджет/исправления | До300 обращений суммарно, включая до3 исправлений одного предложения |
| Память поиска | 3 независимые популяции; до3 демонстраций, кластеры по validation MAP, Boltzmann sampling |
| Форма признаков | JSON: до24 агрегатов и до12 деревьев формул; Llama не пишет SQL/Python; без старых ручных/GNN-признаков |
| До обучения | Проверка на малых раннем/позднем train-снимках и search-строках; не считается оценённым набором |
| Основной evaluator | LightGBM, binary objective,150 boosting rounds,31 лист; подбор через MAP@12 |
| Проверка сопоставимости | Raw/LLM LightGBM на одних парах; те же деревья проходят confirmation и финал |
| История | Только выбранная когорта клиентов, до cutoff, последние180 дней; global counts относятся к этой когорте |
| Ресурсы | 30секунд на вызов функции признаков,4GiB на отдельный evaluator,1800секунд на обучение+оценку |
| Отбор | search→confirmation; итоговые val/test открываются после фиксации программы |

В статье рассматриваются табличные classification/regression, используются другие датасеты,
а лимит30секунд относится к полной оценке. Здесь сохраняем оригинальные803377 пар, поэтому
выделяем отдельный больший лимит для полного обучения и4GiB вместо2GiB. По прямому указанию
пользователя весь отбор переведён на LightGBM; XGBoost в этом протоколе не используется.
В Kaggle нет повторного обучения GNN. Это отдельный протокол Structured v1 с проверкой интерфейса, бюджетом успешных наборов
и исправлениями, отличающийся от20 сэмплов статьи. Старые результаты не переиспользуются.
Референсные деревья использовали те же пары, но другие схемы подбора/производные seeds;
GNN дополнительно обучалась на графе. Это сравнение качества конвейеров, не изолированный
эффект способа генерации признаков. Полные старые MAP нельзя подставлять вместо MAP этой выборки.

Источники: [LLM-FE §4.3, Appendix B](https://arxiv.org/html/2503.14434v3),
[код авторов](https://github.com/nikhilsab/LLMFE),
[NF4](https://huggingface.co/docs/transformers/v4.57.1/quantization/bitsandbytes).


In [ ]:
import subprocess, sys
# Не переустанавливаем torch/CUDA из готового окружения Kaggle.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate==1.10.1', 'bitsandbytes==0.48.1',
    'lightgbm==4.6.0', 'sentencepiece==0.2.1', 'duckdb==1.4.1'])


In [ ]:
from pathlib import Path
import os, json, hashlib, time, gc, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

# Оба варианта скачиваются с Hugging Face; различается издатель репозитория.
MODEL_SOURCE = 'unsloth'  # 'unsloth' без токена; 'hf' официальный Meta с доступом/HF_TOKEN
MODEL_SOURCES = {
    'hf': {'repo_id':'meta-llama/Llama-3.1-8B-Instruct', 'requires_token':True},
    'unsloth': {'repo_id':'unsloth/Meta-Llama-3.1-8B-Instruct', 'requires_token':False},
}
if MODEL_SOURCE not in MODEL_SOURCES:
    raise ValueError("MODEL_SOURCE должен быть 'hf' или 'unsloth'")
MODEL_ID = MODEL_SOURCES[MODEL_SOURCE]['repo_id']

def resolve_auth(source, environ, load_secret):
    if source not in MODEL_SOURCES:
        raise ValueError("MODEL_SOURCE должен быть 'hf' или 'unsloth'")
    if not MODEL_SOURCES[source]['requires_token']:
        # False запрещает также автоматическое использование старого токена из HF-кэша.
        return False
    value = environ.get('HF_TOKEN')
    if not value:
        try:
            value = load_secret()
        except Exception:
            value = None
    if not value:
        raise RuntimeError('Для hf нужны одобренный доступ Meta и Kaggle Secret HF_TOKEN; либо выберите unsloth')
    return value

# При повторе можно указать SHA из предыдущего run_config.json; пустая строка разрешит текущую ревизию.
MODEL_REVISION = ''
WORK = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

CONFIG = {
    'protocol': 'hm_llmfe_llama31_8b_lightgbm_structured_fulltrain_v1', 'seed':42,
    'search_backend':'lightgbm',
    'model_id':MODEL_ID, 'model_source':MODEL_SOURCE,
    'quantization':'bnb NF4 4bit/double_quant, FP16 compute',
    'temperature':0.8, 'top_p':1.0, 'top_k':0,
    'islands':3, 'programs_per_prompt':3, 'max_demonstrations':3,
    'target_successful_distinct':20, 'max_llm_calls':300, 'max_repairs_per_proposal':3,
    'program_interface_version':1, 'feature_interface':'structured_aggregates_and_expression_trees', 'preflight_before_full_training':True,
    'max_input_tokens':6144, 'max_new_tokens':2048,
    'train_pairs':803377, 'train_user_times':50000,
    'search_users':256, 'confirmation_users':512, 'heldout_val_users':1024, 'test_users':2048,
    'cpu_threads':min(4, os.cpu_count() or 1),
    'evaluation_memory_gb':4, 'evaluation_timeout_seconds':1800,
    'feature_call_timeout_seconds':30, 'lightgbm_rounds':150,
    'paper_reproduction':False, 'candidate_protocol':'exact saved ID-GNN seed42 pools',
}


In [ ]:
# Код evaluator встроен в ноутбук: клонировать репозиторий не требуется.
MODULES = {'worker.py': '"""CPU-only bounded evaluator for LLM-authored pandas feature programs."""\nimport os\nif __name__==\'__main__\':\n    os.environ[\'CUDA_VISIBLE_DEVICES\']=\'\'\n    for key in [\'OMP_NUM_THREADS\',\'MKL_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'NUMEXPR_NUM_THREADS\']:\n        os.environ[key]=\'1\'\nimport argparse\nimport ast\nimport hashlib\nimport gc\nimport json\nfrom pathlib import Path\nimport resource\nimport signal\nimport time\nimport traceback\nimport numpy as np\nimport pandas as pd\n\nRAW = \'{"aggregates": []}\'\nDISALLOWED_CALLS={\'open\',\'eval\',\'exec\',\'compile\',\'getattr\',\'setattr\',\'delattr\',\'globals\',\'locals\',\'vars\',\n                  \'input\',\'breakpoint\',\'help\',\'dir\',\'memoryview\',\'type\',\'super\',\'__import__\'}\nDISALLOWED_ATTRIBUTES={\'eval\',\'query\',\'read_csv\',\'read_parquet\',\'read_pickle\',\'read_json\',\'read_sql\',\n                        \'read_excel\',\'read_html\',\'read_feather\',\'load\',\'loads\',\'save\',\'savetxt\',\'savez\',\n                        \'savez_compressed\',\'fromfile\',\'tofile\',\'loadtxt\',\'genfromtxt\',\'memmap\',\'ctypes\',\n                        \'f2py\',\'testing\',\'lib\',\'io\',\'compat\'}\n\n\nfrom structured_features import compile_structured, validate_program\n\ndef check_program(source):\n    return validate_program(json.loads(source),allow_empty=True)\n\ndef compile_program(source):\n    return compile_structured(source)\n\n\ndef feature_error(error,func,X,history):\n    """Actionable model feedback without a long pandas traceback or any labels."""\n    frames=[f for f in traceback.extract_tb(error.__traceback__) if f.filename==\'<llm-features>\']\n    lines=getattr(func,\'_feature_source\',\'\').splitlines()\n    location=\'\'\n    if frames:\n        number=frames[-1].lineno\n        location=\'\\nYour code near line \'+str(number)+\':\\n\'+\'\\n\'.join(\n            f\'{i+1}: {lines[i]}\' for i in range(max(0,number-3),min(len(lines),number+2)))\n    return (f\'{type(error).__name__}: {error}\'+location+\n            \'\\nAvailable INPUT X columns: \'+str(list(X.columns))+\n            \'\\nAvailable INPUT history columns: \'+str(list(history.columns))+\n            \'\\nIntermediate tables may have different columns: inspect your groupby/agg/reset_index/merge. \'\n            \'Overlapping non-key columns get _x/_y suffixes after merge. \'\n            \'History has no raw_u_*/raw_i_* columns. Return only f_* features in unchanged X order.\')\n\n\nclass Data:\n    def __init__(self,root):\n        self.root=Path(root)\n        self.manifest=json.loads((self.root/\'manifest.json\').read_text())\n        self.items=pd.read_parquet(self.root/\'articles.parquet\').set_index(\'article_id\')\n        # XGBoost allocates categorical histograms up to the maximum code. Compact\n        # all static item categories consistently in both X and history.\n        self.category_maps={}\n        for column in self.items:\n            encoded,categories=pd.factorize(self.items[column],sort=True)\n            self.category_maps[column]=categories.tolist()\n            self.items[column]=encoded.astype(\'int32\')\n        self.users=pd.read_parquet(self.root/\'customers.parquet\').set_index(\'customer_id\')\n        self.tx=pd.read_parquet(self.root/\'history.parquet\')\n        self.raw_u=[\'raw_u_\'+c.lower() for c in self.users.columns]\n        self.raw_i=[\'raw_i_\'+c for c in self.items.columns]\n\n    def inputs(self,pairs,cutoff):\n        pairs=pairs[[\'customer_id\',\'article_id\']].reset_index(drop=True)\n        users=self.users.reindex(pairs.customer_id).reset_index(drop=True)\n        items=self.items.reindex(pairs.article_id).reset_index(drop=True)\n        users.columns=self.raw_u\n        items.columns=self.raw_i\n        X=pd.concat([pairs,users,items],axis=1)\n        cutoff=pd.Timestamp(cutoff)\n        history=self.tx[(self.tx.t_dat<=cutoff)&(self.tx.t_dat>cutoff-pd.Timedelta(days=180))].copy()\n        assert (history.t_dat<=cutoff).all()\n        history[\'days_ago\']=(cutoff-history.t_dat).dt.days.astype(\'int32\')\n        history=history.drop(columns=\'t_dat\').merge(self.items.reset_index(),on=\'article_id\',how=\'left\',validate=\'many_to_one\')\n        history=history.merge(self.users[[\'age\']].rename(columns={\'age\':\'customer_age\'}).reset_index(),on=\'customer_id\',how=\'left\',validate=\'many_to_one\')\n        return X,history\n\n\ndef matrix(data,pairs,func,expected=None):\n    parts=[]\n    schema=None\n    for cutoff,frame in pairs.groupby(\'timestamp\',sort=False):\n        X,history=data.inputs(frame,cutoff)\n        # Only safe columns, no labels/targets/timestamps, are passed to the program.\n        def expired(*_):\n            raise TimeoutError(\'Feature function exceeded30 seconds for one snapshot\')\n        signal.signal(signal.SIGALRM,expired)\n        signal.alarm(30)\n        try:\n            out=func(X.copy(deep=True),history.copy(deep=True))\n        except Exception as error:\n            raise ValueError(feature_error(error,func,X,history)) from None\n        finally:\n            signal.alarm(0)\n        if not isinstance(out,pd.DataFrame) or not out.index.equals(X.index):\n            raise ValueError(\'Return a DataFrame with unchanged X.index and one row per pair\')\n        if len(out.columns)>48 or out.columns.duplicated().any():\n            raise ValueError(\'At most48 unique generated feature columns\')\n        for col in out.columns:\n            if not isinstance(col,str) or not col.startswith(\'f_\') or not pd.api.types.is_numeric_dtype(out[col]):\n                raise ValueError(f\'Invalid output column {col!r}: return ONLY numeric f_* features in a separate DataFrame F, not X or intermediate aggregates\')\n            if np.array_equal(out[col].to_numpy(),X.customer_id.to_numpy()) or np.array_equal(out[col].to_numpy(),X.article_id.to_numpy()):\n                raise ValueError(\'IDs are join keys, not numeric features\')\n        raw=X[data.raw_u+data.raw_i]\n        full=pd.concat([raw,out],axis=1)\n        if schema is None:\n            schema=list(full)\n        if list(full)!=schema or expected is not None and schema!=expected:\n            raise ValueError(\'Feature schema changed between snapshots or splits\')\n        values=full.to_numpy(dtype=np.float32)\n        values[~np.isfinite(values)]=np.nan\n        parts.append(pd.DataFrame(values,index=frame.index,columns=schema))\n    values=pd.concat(parts).reindex(pairs.index).to_numpy(dtype=np.float32)\n    return values,schema\n\n\ndef map_at_12(predictions,targets):\n    if len(predictions)!=len(targets):\n        raise ValueError(\'User alignment changed\')\n    hits=np.array([np.isin(p,t) for p,t in zip(predictions,targets)])\n    denominator=np.minimum(12,np.array([len(np.unique(t)) for t in targets]))\n    if (denominator==0).any():\n        raise ValueError(\'Empty target list\')\n    ap=((hits.cumsum(axis=1)/np.arange(1,13))*hits).sum(axis=1)/denominator\n    return float(ap.mean()),ap\n\n\ndef rank(scores,pairs,targets):\n    predictions=[]\n    if list(pairs.customer_id.drop_duplicates())!=list(targets.customer_id):\n        raise ValueError(\'Pool/user order changed\')\n    for _,frame in pairs.assign(score=scores).groupby(\'customer_id\',sort=False):\n        if len(frame)<12 or frame.article_id.duplicated().any():\n            raise ValueError(\'Invalid candidate pool\')\n        predictions.append(frame.sort_values([\'score\',\'article_id\'],ascending=[False,True]).article_id.head(12).to_numpy())\n    return np.stack(predictions)\n\n\ndef limits(memory_gb):\n    size=int(memory_gb*1024**3)\n    resource.setrlimit(resource.RLIMIT_AS,(size,size))\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--data\',required=True)\n    parser.add_argument(\'--directory\',required=True)\n    parser.add_argument(\'--stage\',choices=[\'preflight\',\'search\',\'confirm\',\'val\',\'test\',\'replay_val\',\'replay_test\'],required=True)\n    parser.add_argument(\'--threads\',type=int,default=4)\n    parser.add_argument(\'--memory-gb\',type=float,default=4)\n    parser.add_argument(\'--backend\',choices=[\'xgboost\',\'lightgbm\'],default=\'lightgbm\')\n    args=parser.parse_args()\n    limits(args.memory_gb)\n    tick=time.perf_counter()\n    destination=Path(args.directory)\n    source=(destination/\'program.py\').read_text()\n    func=compile_program(source)\n    data=Data(args.data)\n    if args.stage==\'preflight\':\n        train=pd.read_parquet(data.root/\'train_pairs.parquet\')\n        names=None\n        checked=[]\n        for timestamp in [train.timestamp.min(),train.timestamp.max()]:\n            sample=train[train.timestamp==timestamp].head(256).copy().reset_index(drop=True)\n            _,names=matrix(data,sample,func,expected=names)\n            checked.append({\'kind\':\'train\',\'cutoff\':str(timestamp),\'pairs\':len(sample)})\n        sample=pd.read_parquet(data.root/\'search_pairs.parquet\').head(256).reset_index(drop=True)\n        _,names=matrix(data,sample,func,expected=names)\n        checked.append({\'kind\':\'search_validation\',\'pairs\':len(sample)})\n        if len(names)==len(data.raw_u+data.raw_i):\n            raise ValueError(\'Program generated no new features\')\n        result={\'stage\':\'preflight\',\'passed\':True,\'smoke\':True,\'backend\':args.backend,\n                \'program_sha256\':hashlib.sha256(source.encode()).hexdigest(),\n                \'checked\':checked,\'features\':names,\'seconds\':time.perf_counter()-tick,\n                \'counted_as_successful_set\':False}\n        (destination/\'preflight_metrics.json\').write_text(json.dumps(result,indent=2))\n        print(json.dumps(result))\n        return\n    model_path=destination/(\'model.json\' if args.backend==\'xgboost\' else \'model.txt\')\n    if args.stage==\'search\' and not model_path.exists():\n        pairs=pd.read_parquet(data.root/\'train_pairs.parquet\')\n        values,names=matrix(data,pairs,func)\n        if source!=RAW and len(names)==len(data.raw_u+data.raw_i):\n            raise ValueError(\'Program generated no new features\')\n        categorical=set(data.raw_i+[\'raw_u_club_member_status\',\'raw_u_fashion_news_frequency\'])\n        kinds=[\'c\' if n in categorical else \'q\' for n in names]\n        for i,kind in enumerate(kinds):\n            if kind==\'c\':\n                values[values[:,i]<0,i]=np.nan\n        history={}\n        if args.backend==\'xgboost\':\n            import xgboost as xgb\n            dataset=xgb.QuantileDMatrix(values,label=pairs.label.values,feature_names=names,feature_types=kinds,\n                                       enable_categorical=True,nthread=args.threads,max_bin=127)\n            params={\'objective\':\'binary:logistic\',\'eval_metric\':\'logloss\',\'tree_method\':\'hist\',\'device\':\'cpu\',\n                \'max_depth\':6,\'eta\':.05,\'subsample\':.8,\'colsample_bytree\':.9,\'max_bin\':127,\n                \'seed\':42,\'nthread\':args.threads,\'max_cached_hist_node\':64}\n            model=xgb.train(params,dataset,num_boost_round=150,evals=[(dataset,\'train\')],evals_result=history,verbose_eval=False)\n            model.save_model(model_path)\n        else:\n            import lightgbm as lgb\n            params={\'objective\':\'binary\',\'metric\':\'binary_logloss\',\'verbosity\':-1,\'num_threads\':args.threads,\n                    \'learning_rate\':.05,\'num_leaves\':31,\'min_data_in_leaf\':100,\'max_bin\':127,\n                    \'feature_fraction\':.9,\'bagging_fraction\':.8,\'bagging_freq\':1,\'deterministic\':True,\n                    \'force_col_wise\':True,\'seed\':42,\'feature_fraction_seed\':42,\'bagging_seed\':42,\'data_random_seed\':42}\n            dataset=lgb.Dataset(values,label=pairs.label.values,feature_name=names,categorical_feature=[n for n in names if n in categorical])\n            model=lgb.train(params,dataset,num_boost_round=150,valid_sets=[dataset],valid_names=[\'train\'],callbacks=[lgb.record_evaluation(history)])\n            history={\'train\':{\'logloss\':history[\'train\'][\'binary_logloss\']}}\n            model.save_model(str(model_path))\n        (destination/\'schema.json\').write_text(json.dumps({\'features\':names,\'types\':kinds,\'params\':params,\'train_pairs\':len(pairs),\n                                                          \'item_category_maps\':data.category_maps}))\n        (destination/\'learning_curve.json\').write_text(json.dumps(history))\n        del pairs,values,dataset,model\n        gc.collect()\n    if args.backend==\'xgboost\':\n        import xgboost as xgb\n        model=xgb.Booster()\n        model.load_model(model_path)\n        model.set_param({\'nthread\':args.threads,\'device\':\'cpu\'})\n    else:\n        import lightgbm as lgb\n        model=lgb.Booster(model_file=str(model_path))\n    schema=json.loads((destination/\'schema.json\').read_text())\n    split=args.stage.replace(\'replay_\',\'\')\n    pairs=pd.read_parquet(data.root/f\'{split}_pairs.parquet\')\n    targets=pd.read_parquet(data.root/f\'{split}_targets.parquet\')\n    values,names=matrix(data,pairs,func,expected=schema[\'features\'])\n    for i,kind in enumerate(schema[\'types\']):\n        if kind==\'c\':\n            values[values[:,i]<0,i]=np.nan\n    if args.backend==\'xgboost\':\n        scores=model.predict(xgb.DMatrix(values,feature_names=names,feature_types=schema[\'types\'],enable_categorical=True,nthread=args.threads))\n    else:\n        scores=model.predict(values,num_threads=args.threads)\n    predictions=rank(scores,pairs,targets)\n    score,ap=map_at_12(predictions,targets.article_id)\n    if args.stage.startswith(\'replay_\'):\n        assert np.array_equal(predictions,np.load(destination/f\'{split}_predictions.npy\'))\n        assert np.array_equal(ap,np.load(destination/f\'{split}_user_ap.npy\'))\n    else:\n        np.save(destination/f\'{split}_predictions.npy\',predictions)\n        np.save(destination/f\'{split}_user_ap.npy\',ap)\n    metrics={\'map\':score,\'users\':len(targets),\'pairs\':len(pairs),\'seconds\':time.perf_counter()-tick,\n             \'program_sha256\':hashlib.sha256(source.encode()).hexdigest(),\n             \'model_sha256\':hashlib.sha256(model_path.read_bytes()).hexdigest(),\'backend\':args.backend,\n             \'stage\':args.stage,\'threads\':args.threads,\'passed\':True,\'features\':names}\n    (destination/f\'{args.stage}_metrics.json\').write_text(json.dumps(metrics,indent=2))\n    print(json.dumps(metrics))\n\n\nif __name__==\'__main__\':\n    main()\n', 'runtime.py': '"""Notebook host: paper-style islands, local generation callback and CPU evaluator."""\nimport ast\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport time\nimport zipfile\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom worker import RAW, check_program, Data, map_at_12\nfrom structured_features import validate_program, to_sql\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef write_json(path,obj):\n    path=Path(path)\n    temp=path.with_suffix(path.suffix+\'.tmp\')\n    temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef locate_bundle(input_root,working,expected_manifest_sha):\n    input_root=Path(input_root)\n    matches=[]\n    for path in input_root.rglob(\'manifest.json\'):\n        try:\n            if json.loads(path.read_text()).get(\'bundle_id\')==\'hm_llmfe_kaggle_fulltrain_v2\':\n                matches.append(path.parent)\n        except (ValueError,OSError):\n            pass\n    if not matches:\n        archives=list(input_root.rglob(\'hm_llmfe_kaggle_fulltrain_v2.zip\'))\n        if len(archives)!=1:\n            raise FileNotFoundError(\'Add the supplied hm_llmfe_kaggle_fulltrain_v2.zip as a Kaggle Dataset/Input\')\n        destination=Path(working)/\'hm_llmfe_input\'\n        destination.mkdir(parents=True,exist_ok=True)\n        with zipfile.ZipFile(archives[0]) as z:\n            for member in z.infolist():\n                if not (destination/member.filename).resolve().is_relative_to(destination.resolve()):\n                    raise ValueError(\'Archive path outside destination\')\n            z.extractall(destination)\n        matches=[destination]\n    if len(matches)!=1:\n        raise ValueError(\'Several H&M bundles attached; keep exactly one\')\n    root=matches[0]\n    if sha(root/\'manifest.json\')!=expected_manifest_sha:\n        raise ValueError(\'Unexpected bundle manifest/version\')\n    manifest=json.loads((root/\'manifest.json\').read_text())\n    for name,info in manifest[\'files\'].items():\n        if sha(root/name)!=info[\'sha256\']:\n            raise ValueError(\'Data checksum mismatch: \'+name)\n    return root,manifest\n\n\ndef evaluate(data,directory,stage,config):\n    directory=Path(directory)\n    backend=config.get(\'backend\',config.get(\'search_backend\',\'lightgbm\'))\n    result=directory/f\'{stage}_metrics.json\'\n    if result.exists():\n        metrics=json.loads(result.read_text())\n        if metrics.get(\'backend\')!=backend:\n            raise RuntimeError(\'Cached result uses another evaluator; start a new run ID\')\n        if stage==\'preflight\':\n            if metrics[\'program_sha256\']!=sha(directory/\'program.py\'):\n                raise RuntimeError(\'Cached preflight uses another program\')\n            return metrics\n        model_path=directory/(\'model.txt\' if metrics.get(\'backend\')==\'lightgbm\' else \'model.json\')\n        if metrics[\'program_sha256\']!=sha(directory/\'program.py\') or metrics[\'model_sha256\']!=sha(model_path):\n            raise RuntimeError(\'Cached result does not match model/program\')\n        return metrics\n    command=[sys.executable,str(Path(__file__).with_name(\'worker.py\')),\'--data\',str(data),\n             \'--directory\',str(directory),\'--stage\',stage,\'--threads\',str(1 if stage.startswith(\'replay_\') else config[\'cpu_threads\']),\n             \'--memory-gb\',str(config[\'evaluation_memory_gb\']),\'--backend\',backend]\n    env={k:v for k,v in os.environ.items() if not any(s in k.upper() for s in [\'TOKEN\',\'SECRET\',\'API_KEY\'])}\n    env.update(CUDA_VISIBLE_DEVICES=\'\',OMP_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\',NUMEXPR_NUM_THREADS=\'1\')\n    with (directory/f\'{stage}.log\').open(\'w\') as log:\n        try:\n            process=subprocess.run(command,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=config[\'evaluation_timeout_seconds\'])\n        except subprocess.TimeoutExpired:\n            raise TimeoutError(f"Evaluation exceeded {config[\'evaluation_timeout_seconds\']} seconds") from None\n    if process.returncode or not result.exists():\n        raise RuntimeError((directory/f\'{stage}.log\').read_text()[-2000:])\n    return json.loads(result.read_text())\n\n\ndef source_from_response(text):\n    blocks=re.findall(r\'```(?:json)?\\s*\\n(.*?)```\',text,re.S|re.I)\n    source=blocks[0] if blocks else text\n    return json.dumps(validate_program(json.loads(source)),sort_keys=True,indent=2)+\'\\n\'\n\n\ndef source_key(source):\n    return hashlib.sha256(json.dumps(validate_program(json.loads(source)),sort_keys=True).encode()).hexdigest()\n\n\ndef choose_demonstrations(memory,rng):\n    clusters={}\n    for item in memory:\n        clusters.setdefault(item[\'score\'],[]).append(item)\n    scores=np.array(list(clusters),dtype=float)\n    temperature=.1*(1-(len(memory)%10000)/10000)\n    weights=np.exp((scores-scores.max())/temperature)\n    weights/=weights.sum()\n    count=min(3,len(clusters))\n    selected=rng.choice(len(scores),size=count,replace=False,p=weights)\n    return [clusters[scores[i]][int(rng.integers(len(clusters[scores[i]])))] for i in selected]\n\n\ndef prompt(data,memory,seed):\n    demo=choose_demonstrations(memory,np.random.default_rng(seed))\n    prior=[{\'search_map_fraction\':t[\'score\'],\'program\':json.loads((Path(t[\'directory\'])/\'program.py\').read_text())} for t in demo]\n    return \'\'\'Choose useful features for H&M clothing recommendations. Return ONE JSON object only.\nDo NOT write SQL or Python. The host compiles your feature descriptions, handles joins and preserves rows.\nGoal: predict purchases in the next7days, maximize validation MAP@12 through LightGBM.\nHistory contains only past180 days before each cutoff, for a fixed customer cohort (not full population).\nInterface: {"aggregates":[...],"derived":[...]}. Maximum24 aggregates and12 derived formulas.\nStart compact: about3-6 aggregates and1-3 derived formulas; limits remain24/12. Entire JSON <=2048 tokens.\nAGGREGATE fields: name, keys, op, column, window. Name: unique f_lowercase.\nkeys: ["customer_id"], ["article_id"], [one clothing category], or ["customer_id", article/category].\nClothing categories: product_code,product_type_no,graphical_appearance_no,colour_group_code,\nperceived_colour_value_id,perceived_colour_master_id,department_no,index_group_no,section_no,garment_group_no.\nAllowed op: count,count_distinct,sum,avg,min,max,stddev_pop,median.\ncount MUST use column="*". count_distinct may use customer_id/article_id/sales_channel_id/clothing category\nor price/days_ago/customer_age. All OTHER operations may use ONLY price,days_ago,customer_age.\nwindow: integer1..180 days. Default180. price=normalized transaction price, days_ago=days before cutoff.\nHistory is purchases, not views. Time since MOST RECENT purchase uses min on days_ago, not max.\nNo raw_* fields inside aggregates. sales_channel_id is not a candidate grouping key.\nDERIVED fields: name, expr. Expr is a number, {"ref":"existing_feature"}, or {"op":...,"args":[...]}.\nUnary op (one arg): abs,log1p,sqrt,negate. Binary op (two args): add,subtract,multiply,divide,min,max,coalesce.\nReferences: aggregate names, EARLIER derived names and raw static fields ONLY. No unknown/future names.\nRaw static fields: raw_u_age,raw_u_fn,raw_u_active,raw_u_club_member_status,raw_u_fashion_news_frequency,\nand raw_i_ + one clothing category name listed above. No raw_i_fashion_news_frequency exists.\nConstants finite +/-1000000; expression depth<=6, nodes<=64. Divide by0 returns missing.\nlog1p/sqrt clip negative inputs to0. Missing aggregate groups remain missing; use coalesce if desired.\nFORMAT EXAMPLE ONLY (choose your own useful complete set):\n{"aggregates":[{"name":"f_example","keys":["customer_id"],"op":"count","column":"*","window":30}],\n "derived":[{"name":"f_example_log","expr":{"op":"log1p","args":[{"ref":"f_example"}]}}]}\nThe host appends the same15 raw fields; do not copy them as new features without a useful transformation.\nAll803377 training pairs, LightGBM150 rounds/31 leaves; search fixed256 users/6594 candidates.\nEach proposal is a complete set, not a patch. Test is not available during selection.\nPreviously measured sets in this population (empty aggregates is raw control):\n\'\'\'+json.dumps(prior,separators=(\',\',\':\'))\n\n\ndef budget_summary(trials,config):\n    measured=sum(t[\'status\']==\'valid\' for t in trials)\n    return {\'llm_calls\':len(trials),\'repair_calls\':sum(t.get(\'repair\',0)>0 for t in trials),\n            \'new_proposals\':len({t[\'proposal\'] for t in trials}),\n            \'rejected\':sum(t[\'status\']==\'rejected\' for t in trials),\n            \'duplicates\':sum(t[\'status\']==\'duplicate\' for t in trials),\n            \'evaluated_distinct\':measured,\'target\':config[\'target_successful_distinct\'],\n            \'max_llm_calls\':config[\'max_llm_calls\'],\'max_repairs_per_proposal\':config[\'max_repairs_per_proposal\'],\n            \'target_met\':measured>=config[\'target_successful_distinct\'],\n            \'budget_exhausted\':len(trials)>=config[\'max_llm_calls\'],\n            \'prompt_tokens\':sum(t.get(\'prompt_tokens\',0) for t in trials),\n            \'llm_seconds\':sum(t.get(\'llm_seconds\',0) for t in trials),\n            \'generated_tokens\':sum(t.get(\'generated_tokens\',0) for t in trials)}\n\n\ndef progress(out,trials,config):\n    out=Path(out)\n    write_json(out/\'trials.json\',trials)\n    summary=budget_summary(trials,config)\n    write_json(out/\'search_state.json\',summary)\n    valid=[t for t in trials if t[\'status\']==\'valid\']\n    fig,axes=plt.subplots(1,3,figsize=(15,4))\n    if valid:\n        x=[t[\'sample\'] for t in valid]\n        y=100*np.array([t[\'score\'] for t in valid])\n        axes[0].scatter(x,y,label=\'Candidate\')\n        axes[0].step(x,np.maximum.accumulate(y),where=\'post\',label=\'Best search\')\n        axes[0].legend()\n        axes[1].plot(np.arange(1,len(valid)+1),np.maximum.accumulate(y),marker=\'.\')\n        axes[2].hist(y,bins=min(10,len(valid)))\n    axes[0].set(xlabel=\'LLM call (repairs included)\',ylabel=\'Search MAP@12 (%)\')\n    axes[1].set(xlabel=\'Distinct evaluated sets\',ylabel=\'Best search MAP@12 (%)\')\n    axes[2].set(xlabel=\'Search MAP@12 (%)\',ylabel=\'Evaluated programs\')\n    fig.suptitle(\'Search only: fixed256 users/candidates — not test\')\n    fig.tight_layout();fig.savefig(out/\'search_progress.png\',dpi=150);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    labels=[\'valid\',\'rejected\',\'duplicate\']\n    axes[0].bar(labels,[sum(t[\'status\']==s for t in trials) for s in labels])\n    axes[0].set_ylabel(\'Generated programs\')\n    for t in valid:\n        p=Path(t[\'directory\'])/\'learning_curve.json\'\n        history=json.loads(p.read_text())[\'train\'][\'logloss\']\n        axes[1].plot(np.arange(1,len(history)+1),history,label=str(t[\'sample\']),alpha=.7)\n    axes[1].set(xlabel=\'Boosting round\',ylabel=\'Train binary logloss\')\n    if valid:\n        axes[1].legend(ncol=4,fontsize=7)\n    fig.tight_layout();fig.savefig(out/\'training_and_rejections.png\',dpi=150);plt.close(fig)\n    lines=[\'# Search progress\',\'\',\n           f"LLM calls: {len(trials)}/{summary[\'max_llm_calls\']}; evaluated distinct: {len(valid)}/{summary[\'target\']}; repair calls: {summary[\'repair_calls\']}.",\n           \'Search validation only; no final test results yet.\',\'\',\n           \'| Call | Proposal | Island | Repair | Parent call | Status | MAP@12, % |\',\'|---|---:|---:|---:|---:|---|---:|\']\n    for t in trials:\n        value=f"{100*t[\'score\']:.5f}" if t[\'status\']==\'valid\' else \'—\'\n        lines.append(f"| {t[\'sample\']} | {t[\'proposal\']} | {t[\'island\']} | {t[\'repair\']} | {t.get(\'parent_call\') or \'—\'} | {t[\'status\']} | {value} |")\n    lines+=[\'\',\'![Search](search_progress.png)\',\'![Training](training_and_rejections.png)\']\n    (out/\'progress.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n\n\ndef repair_prompt(base,parent,error):\n    return (base+\'\\nREPAIR: return the COMPLETE corrected JSON. Fix ALL errors listed below, not just the first.\\n\'\n            +\'Previous full response:\\n\'+parent+\'\\nErrors:\\n\'+error[-6500:]\n            +\'\\nNo SQL/Python text. Use only the documented operations and namespaces. \'\n            +\'You may remove invalid features and dependent formulas. Keep the set compact. \'\n            +\'For duplicate rejection choose a different useful complete set.\')\n\n\ndef search(data,out,config,generate):\n    data,out=Path(data),Path(out)\n    out.mkdir(parents=True,exist_ok=True)\n    if (out/\'selected.json\').exists():\n        return json.loads((out/\'selected.json\').read_text())\n    control=out/\'raw_control\';control.mkdir(exist_ok=True)\n    (control/\'program.py\').write_text(RAW)\n    baseline=evaluate(data,control,\'search\',config)\n    raw={\'directory\':str(control),\'score\':baseline[\'map\']}\n    memories=[[raw.copy()] for _ in range(3)]\n    trials=[]\n    seen=set()\n    target=config[\'target_successful_distinct\']\n    max_calls=config[\'max_llm_calls\']\n    max_repairs=config[\'max_repairs_per_proposal\']\n    if target<1 or max_calls<1 or max_repairs<0:\n        raise ValueError(\'Positive successful-set/request budgets and nonnegative repair budget required\')\n    proposal=0\n    repair=0\n    parent=None\n    while len(seen)<target and len(trials)<max_calls:\n        index=len(trials)\n        batch=proposal//3\n        batchdir=out/f\'batch_{batch:02d}\';batchdir.mkdir(exist_ok=True)\n        if not (batchdir/\'meta.json\').exists():\n            island=int(np.random.default_rng(42+batch).integers(3))\n            text=prompt(data,memories[island],42+batch)\n            (batchdir/\'prompt.txt\').write_text(text)\n            write_json(batchdir/\'meta.json\',{\'island\':island,\'new_proposals_per_prompt\':3,\'seed\':42+batch})\n        meta=json.loads((batchdir/\'meta.json\').read_text());island=meta[\'island\']\n        directory=out/\'candidates\'/f\'candidate_{index+1:03d}\';directory.mkdir(parents=True,exist_ok=True)\n        if (directory/\'trial.json\').exists():\n            trial=json.loads((directory/\'trial.json\').read_text())\n        else:\n            trial={\'sample\':index+1,\'call\':index+1,\'proposal\':proposal+1,\'repair\':repair,\n                   \'parent_call\':parent[\'sample\'] if parent is not None else None,\n                   \'island\':island,\'batch\':batch,\'directory\':str(directory)}\n            tick=time.perf_counter()\n            failure_stage=\'generation\'\n            try:\n                response_path=directory/\'response.json\'\n                text=(batchdir/\'prompt.txt\').read_text()\n                if repair:\n                    parent_path=Path(parent[\'directory\'])/\'response.json\'\n                    previous=json.loads(parent_path.read_text()).get(\'text\',\'\') if parent_path.exists() else \'\'\n                    # Keep repair prompts within context: prior code + error + raw example,\n                    # rather than appending to three potentially long memory programs.\n                    text=repair_prompt(prompt(data,[raw],42+proposal),previous,parent[\'error\'])\n                (directory/\'prompt.txt\').write_text(text)\n                if response_path.exists():\n                    response=json.loads(response_path.read_text())\n                else:\n                    request_path=directory/\'request.json\'\n                    if request_path.exists():\n                        raise RuntimeError(\'Previous request was interrupted; its budget slot is consumed. Use a new request for repair.\')\n                    write_json(request_path,{**trial,\'seed\':42+index})\n                    print(f"LLM call{index+1}/{max_calls}, measured{len(seen)}/{target}, island{island}, repair{repair}/{max_repairs}",flush=True)\n                    response=generate(text,42+index)\n                    write_json(response_path,response)\n                for name in [\'prompt_tokens\',\'generated_tokens\',\'seconds\']:\n                    trial[\'llm_seconds\' if name==\'seconds\' else name]=response.get(name,0)\n                trial[\'truncated\']=bool(response.get(\'truncated\'))\n                failure_stage=\'parse_and_validate\'\n                if response.get(\'truncated\'):\n                    raise ValueError(\'Program truncated at output token limit\')\n                code=source_from_response(response[\'text\'])\n                (directory/\'program.py\').write_text(code)\n                (directory/\'program.json\').write_text(code)\n                write_json(directory/\'compiled_sql.json\',to_sql(json.loads(code)))\n                key=source_key(code);trial[\'program_key\']=key\n                if key in seen:\n                    trial.update(status=\'duplicate\',error=\'Duplicate canonical structured specification\')\n                else:\n                    failure_stage=\'preflight\'\n                    print(f"call{index+1}: Structured specification validated; checking small time-safe snapshots (not a measured set)",flush=True)\n                    preflight=evaluate(data,directory,\'preflight\',config)\n                    trial[\'preflight\']=preflight\n                    failure_stage=\'feature_train_score\'\n                    print(f"call{index+1}: preflight passed; starting LightGBM on{config.get(\'train_pairs\',\'configured\')} training pairs",flush=True)\n                    metrics=evaluate(data,directory,\'search\',config)\n                    trial.update(status=\'valid\',score=metrics[\'map\'],metrics=metrics)\n            except Exception as error:\n                trial.update(status=\'rejected\',error=str(error),failure_stage=failure_stage)\n            trial[\'elapsed_seconds\']=time.perf_counter()-tick\n            write_json(directory/\'trial.json\',trial)\n        trials.append(trial)\n        if trial[\'status\']==\'valid\':\n            seen.add(trial[\'program_key\'])\n            if trial[\'score\']>max(t[\'score\'] for t in memories[island]):\n                memories[island].append(trial)\n        write_json(out/\'memory.json\',memories)\n        progress(out,trials,config)\n        print(f"call{index+1}/{max_calls}, island{island}, repair{repair}: {trial[\'status\']}; measured={len(seen)}/{target}",flush=True)\n        if trial[\'status\']!=\'valid\':\n            print(f"REJECTION call{index+1}: stage={trial.get(\'failure_stage\',\'duplicate\')}, "\n                  f"generated_tokens={trial.get(\'generated_tokens\')}, truncated={trial.get(\'truncated\')}",flush=True)\n            print(trial.get(\'error\',\'No error text saved\'),flush=True)\n        else:\n            print(f"ACCEPTED call{index+1}: search MAP@12={100*trial[\'score\']:.6f}%",flush=True)\n        if trial[\'status\']!=\'valid\' and repair<max_repairs:\n            repair+=1\n            parent=trial\n        else:\n            proposal+=1\n            repair=0\n            parent=None\n    write_json(out/\'search_complete.json\',budget_summary(trials,config))\n    # Raw control can win; selecting a harmful LLM program is not forced.\n    options=[raw]+[t for t in trials if t[\'status\']==\'valid\']\n    options=sorted(options,key=lambda t:t[\'score\'],reverse=True)[:3]\n    confirmations=[]\n    for t in options:\n        metrics=evaluate(data,Path(t[\'directory\']),\'confirm\',config)\n        confirmations.append({\'directory\':t[\'directory\'],\'score\':metrics[\'map\'],\'metrics\':metrics})\n        write_json(out/\'confirmation.json\',confirmations)\n    best=max(confirmations,key=lambda t:t[\'score\'])\n    selected={**best,\'selection\':\'512 disjoint validation users; frozen before heldout val/test\',\n              \'selected_raw_control\':Path(best[\'directory\'])==control,\n              \'search_budget\':budget_summary(trials,config)}\n    write_json(out/\'selected.json\',selected)\n    return selected\n\n\ndef final(data,out,config):\n    data,out=Path(data),Path(out)\n    selected=json.loads((out/\'selected.json\').read_text())\n    winner_name=\'Search winner: raw control (NO LLM features)\' if selected[\'selected_raw_control\'] else \'LLM-FE LightGBM\'\n    choices={\'Raw LightGBM\':out/\'raw_control\',winner_name:Path(selected[\'directory\'])}\n    metrics={}\n    for name,directory in choices.items():\n        settings={**config,\'backend\':\'lightgbm\'}\n        metrics[name]={}\n        for split in [\'val\',\'test\']:\n            original=evaluate(data,directory,split,settings)\n            replay=evaluate(data,directory,\'replay_\'+split,settings)\n            assert original[\'map\']==replay[\'map\'] and replay[\'passed\']\n            metrics[name][split]=original\n    # Native reference is never included in prompts or model selection.\n    references={\'Manual features → LightGBM\':\'manual_lightgbm\',\n                \'GNN → LightGBM\':\'gnn_lightgbm\',\n                \'ID-GNN (graph training budget)\':\'idgnn\'}\n    for name,label in references.items():\n        metrics[name]={}\n        for split in [\'val\',\'test\']:\n            targets=pd.read_parquet(data/f\'{split}_targets.parquet\')\n            value,_=map_at_12(np.load(data/f\'{split}_{label}_reference.npy\'),targets.article_id)\n            metrics[name][split]={\'map\':value,\'reference\':True}\n    frame=pd.DataFrame([{\'model\':name,\'val_map\':r[\'val\'][\'map\'],\'test_map\':r[\'test\'][\'map\']} for name,r in metrics.items()])\n    frame.to_csv(out/\'comparison.csv\',index=False)\n    fig,ax=plt.subplots(figsize=(10,4))\n    for i,split in enumerate([\'val\',\'test\']):\n        bars=ax.barh(np.arange(len(frame))+(i-.5)*.32,100*frame[split+\'_map\'],height=.32,label=split)\n        ax.bar_label(bars,fmt=\'%.3f\',padding=3,fontsize=8)\n    ax.set_yticks(np.arange(len(frame)),frame.model);ax.set_xlabel(\'MAP@12 (%) — identical fixed candidate pools\');ax.legend()\n    ax.set_xlim(0,max(.1,float(frame[[\'val_map\',\'test_map\']].to_numpy().max())*118))\n    if config.get(\'smoke\'):\n        fig.suptitle(\'SYNTHETIC SMOKE — not Llama results\')\n    fig.tight_layout();fig.savefig(out/\'comparison.png\',dpi=150,bbox_inches=\'tight\');plt.close(fig)\n    aps={name:np.load(path/\'test_user_ap.npy\') for name,path in choices.items()}\n    truth=pd.read_parquet(data/\'test_targets.parquet\')\n    for name,label in references.items():\n        aps[name]=map_at_12(np.load(data/f\'test_{label}_reference.npy\'),truth.article_id)[1]\n    pd.DataFrame({\'customer_id\':truth.customer_id.values,**aps}).to_csv(out/\'test_user_ap.csv\',index=False)\n    fig,axes=plt.subplots(1,2,figsize=(13,4))\n    for name,values in aps.items():\n        axes[0].hist(100*values[values>0],bins=np.linspace(0,100,21),histtype=\'step\',label=name)\n    axes[0].set(xlabel=\'Positive user AP@12 (%)\',ylabel=\'Users\',yscale=\'log\');axes[0].legend(fontsize=7)\n    axes[1].barh(list(aps),[100*np.mean(a==0) for a in aps.values()]);axes[1].set(xlabel=\'Test users without hits (%)\',xlim=(0,100))\n    if config.get(\'smoke\'):\n        fig.suptitle(\'SYNTHETIC SMOKE — not Llama results\')\n    fig.tight_layout();fig.savefig(out/\'ap_histograms.png\',dpi=150,bbox_inches=\'tight\');plt.close(fig)\n    delta=aps[winner_name]-aps[\'Raw LightGBM\']\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    axes[0].bar([\'LLM better\',\'Tie\',\'Raw better\'],[(delta>0).sum(),(delta==0).sum(),(delta<0).sum()])\n    if np.any(delta!=0):\n        axes[1].hist(100*delta[delta!=0],bins=30);axes[1].set_yscale(\'log\')\n    axes[1].set(xlabel=\'LLM minus raw AP (percentage points)\',ylabel=\'Users; ties omitted\')\n    fig.tight_layout();fig.savefig(out/\'paired_comparison.png\',dpi=150);plt.close(fig)\n    fig,axes=plt.subplots(1,2,figsize=(11,4))\n    for name in [\'Raw LightGBM\',winner_name]:\n        curve=json.loads((choices[name]/\'learning_curve.json\').read_text())[\'train\'][\'logloss\']\n        axes[0].plot(np.arange(1,len(curve)+1),curve,label=name)\n    axes[0].set(xlabel=\'Boosting round\',ylabel=\'Train binary logloss\');axes[0].legend()\n    delta_lgb=aps[winner_name]-aps[\'Raw LightGBM\']\n    axes[1].bar([\'LLM better\',\'Tie\',\'Raw better\'],[(delta_lgb>0).sum(),(delta_lgb==0).sum(),(delta_lgb<0).sum()])\n    axes[1].set_title(\'LightGBM: paired test AP\')\n    fig.tight_layout();fig.savefig(out/\'lightgbm_comparison.png\',dpi=150);plt.close(fig)\n    trials=json.loads((out/\'trials.json\').read_text())\n    report={\'complete\':True,\'metrics\':metrics,\'selected\':selected,\'config\':config,\n            \'search_budget\':budget_summary(trials,config),\n            \'smoke\':bool(config.get(\'smoke\',False)),\n            \'generated_programs\':sum((Path(t[\'directory\'])/\'response.json\').exists() for t in trials),\n            \'evaluated_distinct\':sum(t[\'status\']==\'valid\' for t in trials),\n            \'llm_feature_model_selected\':not selected[\'selected_raw_control\'],\n            \'llm_program_quality_measured\':any(t[\'status\']==\'valid\' for t in trials),\n            \'independent_full_replay\':True,\'coverage\':\'bounded exported subset, not entire H&M\',\n            \'paper_reproduction\':False}\n    write_json(out/\'report.json\',report)\n    lines=[\'# Llama3.1-8B → Python features → LightGBM: bounded H&M\',\'\',\n           \'Adaptation of LLM-FE; not a numerical reproduction of the paper. All MAP values in percent.\',\n           \'One seed42,180-day selected-cohort history, frozen shared ID-GNN pools.\',\n           \'LightGBM selects features throughout search and confirmation; all new trees use identical803377 training pairs.\',\n           \'Manual and GNN→LightGBM references use those same803377 supervised pairs and these exact evaluation rows/pools.\',\n           \'Tuning protocols, derived seeds and historical representations differ; GNN also used upstream graph training.\',\n           \'Compare pipeline quality on this subset, not differences in isolated feature quality or full-dataset MAP.\',\n           \'Selection uses only search256 + confirmation512; heldout val1024 and test2048 used after freezing.\',\n           \'Native ID-GNN reference used more training data and is contextual, not a matched-budget comparison.\',\n           f"LLM calls{len(trials)}/{config[\'max_llm_calls\']}; evaluated distinct{report[\'evaluated_distinct\']}/{config[\'target_successful_distinct\']}; target met: {report[\'search_budget\'][\'target_met\']}. Raw control selected: {selected[\'selected_raw_control\']}.",\n           \'All raw/LLM heldout predictions reproduced from saved models in separate CPU processes with1 thread.\',\n           \'\', \'| Model | Validation MAP, % | Test MAP, % |\',\'|---|---:|---:|\']\n    if config.get(\'smoke\'):\n        lines=[\'# SYNTHETIC SMOKE — not Llama results\',\'\',\n               \'This report tests software using a synthetic generator and tiny data. No real Llama inference or quality experiment.\', \'\']+lines\n    if selected[\'selected_raw_control\']:\n        lines.insert(2,\'RAW CONTROL SELECTED: the search-winner row contains NO LLM features. Do not report its MAP as the quality of LLM-generated features.\')\n    lines += [f"| {r.model} | {100*r.val_map:.5f} | {100*r.test_map:.5f} |" for r in frame.itertuples()]\n    lines += [\'\',f"Selected program: `{Path(selected[\'directory\']).relative_to(out)}/program.py`.",\n              \'![MAP](comparison.png)\',\'![AP](ap_histograms.png)\',\'![Paired](paired_comparison.png)\',\n              \'![Search](search_progress.png)\',\'![Training](training_and_rejections.png)\',\n              \'![LightGBM](lightgbm_comparison.png)\']\n    (out/\'report.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    return report\n', 'sql_features.py': '"""Restricted SQL aggregates. The host owns tables, joins, time filtering and row order."""\nimport json\nimport re\nimport duckdb\nimport pandas as pd\n\nITEMS = [\'product_code\',\'product_type_no\',\'graphical_appearance_no\',\'colour_group_code\',\n         \'perceived_colour_value_id\',\'perceived_colour_master_id\',\'department_no\',\n         \'index_group_no\',\'section_no\',\'garment_group_no\']\nKEYS = {\'customer_id\',\'article_id\',*ITEMS}\nCOLUMNS = KEYS | {\'price\',\'days_ago\',\'sales_channel_id\',\'customer_age\'}\nFUNCTIONS = {\'count\',\'sum\',\'avg\',\'min\',\'max\',\'stddev_pop\',\'stddev_samp\',\'variance\',\'median\',\n             \'quantile_cont\',\'mode\',\'coalesce\',\'nullif\',\'abs\',\'sqrt\',\'ln\',\'log\',\'log2\',\'log10\',\n             \'exp\',\'pow\',\'power\',\'round\',\'floor\',\'ceil\',\'cast\',\'greatest\',\'least\',\'approx_count_distinct\'}\nWORDS = {\'case\',\'when\',\'then\',\'else\',\'end\',\'distinct\',\'filter\',\'where\',\'and\',\'or\',\n         \'not\',\'is\',\'null\',\'between\',\'in\',\'true\',\'false\',\'as\',\'double\',\'float\',\'integer\',\'bigint\'}\nTOKEN = re.compile(r\'\\s+|[A-Za-z][A-Za-z_0-9]*|\\d+(?:\\.\\d+)?|<=|>=|<>|!=|[(),+*/%=<>-]\')\n\n\nRAW = {\'raw_u_age\',\'raw_u_fn\',\'raw_u_active\',\'raw_u_club_member_status\',\'raw_u_fashion_news_frequency\',\n       *(\'raw_i_\'+name for name in ITEMS)}\n\n\ndef validate_expression(sql,columns=None,aggregate=True):\n    columns=COLUMNS if columns is None else columns\n    if not isinstance(sql,str) or not sql or len(sql)>1000:\n        raise ValueError(\'sql must be a nonempty numeric expression <=1000 characters\')\n    tokens=[]; offset=0\n    while offset<len(sql):\n        match=TOKEN.match(sql,offset)\n        if not match:\n            raise ValueError(\'Only numeric SQL expressions; no statements, strings, comments, or qualified names\')\n        offset=match.end()\n        if not match.group().isspace():\n            tokens.append(match.group().lower())\n    for i,token in enumerate(tokens):\n        if re.fullmatch(\'[a-z][a-z_0-9]*\',token):\n            if token not in columns|FUNCTIONS|WORDS:\n                raise ValueError(\'Unknown or forbidden SQL identifier: \'+token)\n            if i+1<len(tokens) and tokens[i+1]==\'(\' and token not in FUNCTIONS|{\'filter\',\'in\',\'and\',\'or\',\'not\',\'where\'}:\n                raise ValueError(\'Function outside allowlist: \'+token)\n    # IDs can be counted, never averaged or used as numerical predictions.\n    masked=re.sub(r\'\\b(?:count|approx_count_distinct)\\s*\\(\\s*(?:distinct\\s+)?(?:customer_id|article_id)\\s*\\)\',\n                  \'count(*)\',sql,flags=re.I) if aggregate else sql\n    if re.search(r\'\\b(customer_id|article_id)\\b\',masked,re.I):\n        raise ValueError(\'IDs only in grouping keys or count(DISTINCT id)\')\n    return \' \'.join(tokens)\n\n\ndef validate_program(program,allow_empty=False):\n    if not isinstance(program,dict) or set(program)-{\'rationale\',\'aggregates\',\'derived\'}:\n        raise ValueError(\'Return JSON with aggregates, derived and optional rationale only\')\n    entries=program.get(\'aggregates\',[]); derived=program.get(\'derived\',[])\n    if not isinstance(entries,list) or not isinstance(derived,list) or len(entries)>24 or len(derived)>12:\n        raise ValueError(\'Maximum24 aggregates and12 derived features, as in local Qwen interface\')\n    if not entries and not derived and not allow_empty:\n        raise ValueError(\'Provide at least one generated feature; raw control is host-only\')\n    names=set(); output=[]\n    for entry in entries:\n        if not isinstance(entry,dict) or set(entry)!={\'name\',\'keys\',\'sql\'}:\n            raise ValueError(\'Each aggregate needs exactly name, keys, sql\')\n        name=entry[\'name\']; keys=entry[\'keys\']\n        if not isinstance(name,str) or not re.fullmatch(\'f_[a-z][a-z0-9_]{0,45}\',name) or name in names:\n            raise ValueError(\'Use unique f_lowercase feature names\')\n        names.add(name)\n        if not isinstance(keys,list) or not 1<=len(keys)<=2 or any(not isinstance(k,str) or k not in KEYS for k in keys) or len(set(keys))!=len(keys):\n            raise ValueError(\'Invalid grouping keys\')\n        if len(keys)==2 and \'customer_id\' not in keys:\n            raise ValueError(\'Two keys must be customer_id + article_id or clothing category\')\n        output.append({\'name\':name,\'keys\':sorted(keys),\'sql\':validate_expression(entry[\'sql\'])})\n    formulas=[]; available=RAW|names\n    for entry in derived:\n        if not isinstance(entry,dict) or set(entry)!={\'name\',\'sql\'}:\n            raise ValueError(\'Each derived feature needs exactly name, sql\')\n        name=entry[\'name\']\n        if not isinstance(name,str) or not re.fullmatch(\'f_[a-z][a-z0-9_]{0,45}\',name) or name in names:\n            raise ValueError(\'Use unique f_lowercase feature names\')\n        formulas.append({\'name\':name,\'sql\':validate_expression(entry[\'sql\'],columns=available,aggregate=False)})\n        names.add(name);available.add(name)\n    return {\'aggregates\':sorted(output,key=lambda e:e[\'name\']),\'derived\':formulas}\n\n\ndef compile_sql(source):\n    program=validate_program(json.loads(source),allow_empty=True)\n\n    def modify_features(X,history):\n        result=pd.DataFrame(index=X.index)\n        if not program[\'aggregates\'] and not program[\'derived\']:\n            return result\n        connection=duckdb.connect(config={\'enable_external_access\':\'false\',\'threads\':\'1\',\n                                         \'memory_limit\':\'512MB\'})\n        try:\n            connection.register(\'history_input\',history)\n            groups={}\n            for entry in program[\'aggregates\']:\n                groups.setdefault(tuple(entry[\'keys\']),[]).append(entry)\n            for keys,entries in groups.items():\n                query=\'SELECT \'+\',\'.join(keys)+\',\'+\',\'.join(e[\'sql\']+\' AS \'+e[\'name\'] for e in entries)\n                query+=\' FROM history_input GROUP BY \'+\',\'.join(keys)\n                aggregate=connection.execute(query).df()\n                left=pd.DataFrame(index=X.index)\n                for key in keys:\n                    left[key]=X[key] if key in {\'customer_id\',\'article_id\'} else X[\'raw_i_\'+key]\n                joined=left.merge(aggregate,on=list(keys),how=\'left\',sort=False,validate=\'many_to_one\')\n                if len(joined)!=len(X):\n                    raise ValueError(\'Host SQL join changed row count\')\n                for entry in entries:\n                    result[entry[\'name\']]=joined[entry[\'name\']].to_numpy()\n            # Derived expressions see ONLY raw fields and previously computed features.\n            frame=pd.concat([X[[c for c in X if c.startswith(\'raw_\')]],result],axis=1)\n            for entry in program[\'derived\']:\n                connection.register(\'derived_input\',frame.reset_index(drop=True))\n                values=connection.execute(\'SELECT \'+entry[\'sql\']+\' AS \'+entry[\'name\']+\' FROM derived_input\').df()\n                connection.unregister(\'derived_input\')\n                if len(values)!=len(X):\n                    raise ValueError(\'Derived SQL must produce one scalar per candidate row\')\n                frame[entry[\'name\']]=values[entry[\'name\']].to_numpy()\n                result[entry[\'name\']]=values[entry[\'name\']].to_numpy()\n        finally:\n            connection.close()\n        return result[[e[\'name\'] for e in program[\'aggregates\']+program[\'derived\']]]\n    modify_features._feature_source=source\n    return modify_features\n', 'structured_features.py': '"""Compile bounded feature descriptions into host-owned SQL; never execute model code."""\nimport json\nimport math\nimport re\nfrom decimal import Decimal\nfrom sql_features import KEYS,RAW,compile_sql\n\nAGGREGATIONS={\'count\',\'count_distinct\',\'sum\',\'avg\',\'min\',\'max\',\'stddev_pop\',\'median\'}\nNUMERIC={\'price\',\'days_ago\',\'customer_age\'}\nCOUNTABLE=KEYS|{\'sales_channel_id\'}|NUMERIC\nUNARY={\'abs\',\'log1p\',\'sqrt\',\'negate\'}\nBINARY={\'add\',\'subtract\',\'multiply\',\'divide\',\'min\',\'max\',\'coalesce\'}\n\n\ndef expression(node,available,depth=0,budget=None):\n    budget=[0] if budget is None else budget\n    budget[0]+=1\n    if depth>6 or budget[0]>64:\n        raise ValueError(\'Expression limit: depth6,64 nodes\')\n    if isinstance(node,(int,float)) and not isinstance(node,bool):\n        if not math.isfinite(node) or abs(node)>1e6:\n            raise ValueError(\'Constants must be finite and within +/-1000000\')\n        return format(Decimal(str(node)),\'f\')\n    if not isinstance(node,dict):\n        raise ValueError(\'Expression: number, {"ref":"existing_feature"}, or {"op":...,"args":[...]}\')\n    if set(node)=={\'ref\'}:\n        if not isinstance(node[\'ref\'],str) or node[\'ref\'] not in available:\n            raise ValueError(\'Unknown reference \'+str(node[\'ref\'])+\'; available: \'+\',\'.join(sorted(available)))\n        return node[\'ref\']\n    if set(node)!={\'op\',\'args\'} or not isinstance(node[\'op\'],str) or not isinstance(node[\'args\'],list):\n        raise ValueError(\'Operation needs exactly op and args\')\n    op=node[\'op\'];args=node[\'args\']\n    if op not in UNARY|BINARY or len(args)!=(1 if op in UNARY else 2):\n        raise ValueError(\'Unary ops: \'+\',\'.join(sorted(UNARY))+\'; binary ops: \'+\',\'.join(sorted(BINARY)))\n    sql=[expression(arg,available,depth+1,budget) for arg in args]\n    a=\'(\'+sql[0]+\')\'\n    if op==\'abs\':return \'ABS(\'+a+\')\'\n    if op==\'log1p\':return \'LN(1 + GREATEST(\'+a+\', 0))\'\n    if op==\'sqrt\':return \'SQRT(GREATEST(\'+a+\', 0))\'\n    if op==\'negate\':return \'-\'+a\n    b=\'(\'+sql[1]+\')\'\n    if op==\'divide\':return a+\' / NULLIF(\'+b+\', 0)\'\n    if op in {\'add\',\'subtract\',\'multiply\'}:return a+{\'add\':\'+\',\'subtract\':\'-\',\'multiply\':\'*\'}[op]+b\n    return {\'min\':\'LEAST\',\'max\':\'GREATEST\',\'coalesce\':\'COALESCE\'}[op]+\'(\'+a+\',\'+b+\')\'\n\n\ndef validate_program(program,allow_empty=False):\n    if not isinstance(program,dict) or set(program)-{\'aggregates\',\'derived\',\'rationale\'}:\n        raise ValueError(\'JSON object with aggregates and derived only (optional rationale)\')\n    aggs=program.get(\'aggregates\',[]);derived=program.get(\'derived\',[])\n    if not isinstance(aggs,list) or not isinstance(derived,list) or len(aggs)>24 or len(derived)>12:\n        raise ValueError(\'Maximum24 aggregates and12 derived formulas\')\n    if not aggs and not derived and not allow_empty:\n        raise ValueError(\'At least one generated feature required\')\n    errors=[]; names=set(); normalized=[]; formulas=[]\n    def name_check(name):\n        if not isinstance(name,str) or not re.fullmatch(\'f_[a-z][a-z0-9_]{0,45}\',name) or name in names:\n            raise ValueError(\'name must be unique f_lowercase (<=48 characters)\')\n    for i,entry in enumerate(aggs):\n        try:\n            if not isinstance(entry,dict) or set(entry)-{\'name\',\'keys\',\'op\',\'column\',\'window\'} or not {\'name\',\'keys\',\'op\'}<=set(entry):\n                raise ValueError(\'Use name, keys, op; optional column and window. No sql field.\')\n            name=entry[\'name\'];name_check(name)\n            keys=entry[\'keys\'];op=entry[\'op\'];column=entry.get(\'column\',\'*\');window=entry.get(\'window\',180)\n            if not isinstance(keys,list) or not 1<=len(keys)<=2 or any(not isinstance(k,str) or k not in KEYS for k in keys) or len(set(keys))!=len(keys) or len(keys)==2 and \'customer_id\' not in keys:\n                raise ValueError(\'keys: one customer_id/article_id/clothing category, or customer_id + article/category\')\n            if not isinstance(op,str) or op not in AGGREGATIONS:\n                raise ValueError(\'op must be \'+\',\'.join(sorted(AGGREGATIONS)))\n            if not isinstance(column,str) or (op==\'count\' and column!=\'*\') or (op==\'count_distinct\' and column not in COUNTABLE) or (op not in {\'count\',\'count_distinct\'} and column not in NUMERIC):\n                raise ValueError(\'count uses column="*"; count_distinct uses a history column; numeric aggregates use price/days_ago/customer_age only. No raw_* fields.\')\n            if not isinstance(window,int) or isinstance(window,bool) or not 1<=window<=180:\n                raise ValueError(\'window must be integer days1..180\')\n            normalized.append({\'name\':name,\'keys\':sorted(keys),\'op\':op,\'column\':column,\'window\':window})\n            names.add(name)\n        except ValueError as error:\n            errors.append(f\'aggregates[{i}]: {error}\')\n    available=RAW|names\n    for i,entry in enumerate(derived):\n        try:\n            if not isinstance(entry,dict) or set(entry)!={\'name\',\'expr\'}:\n                raise ValueError(\'Use exactly name and expr (expression tree), no SQL\')\n            name=entry[\'name\'];name_check(name)\n            expression(entry[\'expr\'],available)\n            formulas.append(entry);names.add(name);available.add(name)\n        except ValueError as error:\n            errors.append(f\'derived[{i}]: {error}\')\n    if errors:\n        raise ValueError(\'Fix ALL listed schema errors:\\n\'+\'\\n\'.join(errors))\n    return {\'aggregates\':sorted(normalized,key=lambda e:e[\'name\']),\'derived\':formulas}\n\n\ndef to_sql(program,allow_empty=False):\n    program=validate_program(program,allow_empty=allow_empty)\n    aggs=[]\n    for entry in program[\'aggregates\']:\n        operation=entry[\'op\'];column=entry[\'column\']\n        sql=\'COUNT(DISTINCT \'+column+\')\' if operation==\'count_distinct\' else operation.upper()+\'(\'+column+\')\'\n        sql+=\' FILTER (WHERE days_ago <= \'+str(entry[\'window\'])+\')\'\n        aggs.append({\'name\':entry[\'name\'],\'keys\':entry[\'keys\'],\'sql\':sql})\n    available=RAW|{entry[\'name\'] for entry in aggs};derived=[]\n    for entry in program[\'derived\']:\n        derived.append({\'name\':entry[\'name\'],\'sql\':expression(entry[\'expr\'],available)})\n        available.add(entry[\'name\'])\n    return {\'aggregates\':aggs,\'derived\':derived}\n\n\ndef compile_structured(source):\n    return compile_sql(json.dumps(to_sql(json.loads(source),allow_empty=True)))\n'}
CODE = WORK / 'llmfe_structured_runtime'
CODE.mkdir(exist_ok=True)
for name, source in MODULES.items():
    (CODE / name).write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE))
import importlib
import worker as feature_worker
importlib.reload(feature_worker)
import runtime as experiment
importlib.reload(experiment)


In [ ]:
DATA, manifest = experiment.locate_bundle(INPUT_ROOT, WORK, '3d042b7fb24e702cb0784a99c6f04f82774e70dc802f710744232cddf016372a')
assert manifest['train_pairs'] == 803377 and manifest['train_user_times'] == 50000
assert manifest['files']['train_pairs.parquet']['sha256'] == manifest['train_source_sha256']
CONFIG['bundle_manifest_sha256'] = '3d042b7fb24e702cb0784a99c6f04f82774e70dc802f710744232cddf016372a'
display(pd.DataFrame([{'split':k,'users':v['users'],'pairs':v['pairs']} for k,v in manifest['splits'].items()]))
print('Training pairs:', manifest['train_pairs'], 'history rows:', manifest['history_rows'])
print('Data checksums: OK')


## Доступ к Llama и фиксация запуска

`unsloth` скачивается анонимно, без чтения Kaggle Secrets и сохранённых HF-токенов.
Это полноразмерная публикация весов; NF4-квантизация выполняется при загрузке одинаково
для обоих источников. Unsloth-библиотека и её отдельное окружение не требуются.
`hf` требует одобрения Meta и `HF_TOKEN`. При403 проверьте доступ, при401 — токен.
Автоматического переключения источника нет. Ревизия публикации фиксируется отдельно;
побитовое совпадение весов двух издателей здесь не проверено. Лицензия Llama сохраняется.
Конфигурация, ревизия модели, версии пакетов, хеши данных и evaluator определяют новый run ID;
старые результаты не перезаписываются. Токен не сохраняется в артефакты и не выводится.


In [ ]:
import torch
from huggingface_hub import HfApi, hf_hub_download
if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU T4 в настройках Kaggle')
gpu = torch.cuda.get_device_properties(0)
if gpu.total_memory < 14 * 1024**3:
    raise RuntimeError('Нужно не менее14GiB GPU RAM; выбранная карта не подходит для этой конфигурации')

def load_kaggle_secret():
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret('HF_TOKEN')

token = resolve_auth(MODEL_SOURCE, os.environ, load_kaggle_secret)
try:
    revision = HfApi(token=token).model_info(MODEL_ID, revision=MODEL_REVISION or 'main').sha
    hf_hub_download(MODEL_ID, 'config.json', revision=revision, token=token)
except Exception:
    hint = 'Internet и существование ревизии' if MODEL_SOURCE == 'unsloth' else 'Internet, HF_TOKEN и одобрение доступа Meta'
    raise RuntimeError('Не удалось прочитать ' + MODEL_ID + '. Проверьте ' + hint) from None

CONFIG['model_revision'] = revision
CONFIG['gpu'] = gpu.name
CONFIG['versions'] = {p:importlib.metadata.version(p) for p in
    ['torch','transformers','accelerate','bitsandbytes','lightgbm','pandas','numpy','pyarrow','duckdb']}
CONFIG['runtime_sha256'] = {name:hashlib.sha256(source.encode()).hexdigest() for name,source in MODULES.items()}
fingerprint = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
RUN_ID = CONFIG['protocol'] + '_' + fingerprint
OUT = WORK / RUN_ID
OUT.mkdir(exist_ok=True)
config_path = OUT / 'run_config.json'
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise RuntimeError('Config mismatch: start a new run directory')
experiment.write_json(config_path, CONFIG)
print('Run:', RUN_ID, '| source:', MODEL_SOURCE, '| model:', MODEL_ID, '| GPU:', gpu.name, '| revision:', revision)


In [ ]:
# Контроль без LLM-признаков: реальное обучение на всех803377 парах.
control = OUT / 'raw_control'
control.mkdir(exist_ok=True)
(control / 'program.py').write_text(experiment.RAW, encoding='utf-8')
baseline = experiment.evaluate(DATA, control, 'search', CONFIG)
print('Raw LightGBM search MAP@12:', 100 * baseline['map'], '%')
print('Control training/evaluation seconds:', baseline['seconds'])


## Загрузка Llama на GPU0

Это инференс, без fine-tuning. NF4 позволяет разместить веса и KV-кеш на одной T4.
Используется float16; BF16 и FlashAttention2 не требуются. CPU-деревья обучаются
в отдельном процессе и не занимают VRAM. Первое скачивание весов может быть долгим.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
set_seed(CONFIG['seed'])
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, token=token, trust_remote_code=False)
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, token=token,
    quantization_config=quantization, torch_dtype=torch.float16, device_map={'':0},
    attn_implementation='sdpa', trust_remote_code=False, low_cpu_mem_usage=True)
model.eval()
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
print('Model GPU footprint GiB:', model.get_memory_footprint()/1024**3)
experiment.write_json(OUT / 'model_runtime.json', {
    'model':MODEL_ID, 'model_source':MODEL_SOURCE, 'revision':revision, 'is_loaded_in_4bit':bool(model.is_loaded_in_4bit),
    'gpu':gpu.name, 'footprint_bytes':int(model.get_memory_footprint()),
    'allocated_vram_bytes':int(torch.cuda.memory_allocated(0))})


In [ ]:
def generate_program(prompt, seed):
    set_seed(seed)
    messages = [
        {'role':'system','content':'You are a feature engineering researcher. Return only JSON feature descriptions using the specified operations. Never write SQL or Python.'},
        {'role':'user','content':prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda:0')
    prompt_tokens = inputs['input_ids'].shape[1]
    if prompt_tokens > CONFIG['max_input_tokens']:
        raise ValueError('Prompt exceeds input budget; no silent context truncation')
    endings = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')]
    endings = list({int(i) for i in endings if i is not None})
    tick = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'],
            do_sample=True, temperature=CONFIG['temperature'], top_p=CONFIG['top_p'], top_k=CONFIG['top_k'],
            num_return_sequences=1, eos_token_id=endings, pad_token_id=tokenizer.pad_token_id, use_cache=True)
    generated = output[0, prompt_tokens:]
    result = {'text':tokenizer.decode(generated, skip_special_tokens=True),
        'seed':seed, 'prompt_tokens':int(prompt_tokens), 'generated_tokens':int(len(generated)),
        'seconds':time.perf_counter()-tick,
        'truncated':len(generated)>=CONFIG['max_new_tokens'] and int(generated[-1]) not in endings}
    del output, inputs, generated
    return result

# b=3: одинаковый prompt, три независимых последовательных сэмпла; это экономит VRAM T4.
# Цель20 различных наборов с LightGBM MAP; максимум300 обращений, включая исправления.
# Одно исходное предложение + до3 исправлений = максимум4 обращения на цепочку.
# Ошибки/дубликаты не увеличивают число оценённых наборов. Raw control тоже не входит в20.
selected = experiment.search(DATA, OUT, CONFIG, generate_program)
print('Selection frozen before test:', selected)
print('Search budget:', selected['search_budget'])
if not selected['search_budget']['target_met']:
    print('ВНИМАНИЕ: бюджет обращений исчерпан, цель20 наборов не достигнута. Итог будет по доступным наборам.')
display(Image(filename=str(OUT / 'search_progress.png')))
display(Image(filename=str(OUT / 'training_and_rejections.png')))


## Программа, выбранная LightGBM → итоговая оценка

Выбранные признаки фиксируются по confirmation LightGBM; test не влияет на выбор. Контроль без новых признаков
тоже может победить. Оба новых LightGBM обучаются на тех же803377 парах, что прежние
ручной и GNN→LightGBM. Все методы оцениваются на одинаковых val/test строках и пулах.
Старые модели не перетренируются: их сохранённые top12 включены в архив данных.
Протоколы подбора/производные seeds различаются, GNN обучалась дополнительно на графе.


In [ ]:
# Генерации больше не нужны; освобождаем VRAM.
del model
gc.collect()
torch.cuda.empty_cache()
report = experiment.final(DATA, OUT, CONFIG)
display(pd.read_csv(OUT / 'comparison.csv').assign(
    val_map=lambda x:100*x.val_map, test_map=lambda x:100*x.test_map))
print('Все новые модели: полный повтор предсказаний из сохранённых весов подтверждён.')


In [ ]:
for name in ['comparison.png','ap_histograms.png','paired_comparison.png','lightgbm_comparison.png']:
    display(Image(filename=str(OUT / name)))
display(Markdown((OUT / 'report.md').read_text()))


In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / (RUN_ID + '_results')), 'zip', root_dir=OUT)
display(FileLink(archive))
print('Скачайте ZIP из Output. Веса Llama и HF_TOKEN в него не включены.')


## Что смотреть после запуска

- `report.md`, `comparison.csv`: MAP на одной и той же небольшой выборке, включая три способа + контроли.
- `comparison.png`, `ap_histograms.png`, `paired_comparison.png`, `lightgbm_comparison.png`: сравнение и распределения.
- `search_progress.png`, `training_and_rejections.png`: поиск, отказы и train logloss; loss не выбирает победителя.
- `candidates/candidate_*/program.json` (дубликат в `program.py` для совместимости хранения), `response.json`, `trial.json`: описания признаков, ответы и причины отказов; `compiled_sql.json` — SQL, созданный host.
- `selected.json`, `confirmation.json`: проверяемая история выбора только по validation.
- `model.json`/`model.txt`, `schema.json`, `*_metrics.json`, `*_predictions.npy`: новые деревья и независимый повтор.
- `run_config.json`, `model_runtime.json`: версия Llama, квантизация, пакеты, хеши кода/данных.

LLM не пишет исполняемый Python. Все выражения компилирует host из проверенного дерева операций;
DuckDB не имеет внешнего доступа. Запросы выполняются в отдельном ограниченном CPU-процессе. LLM не получает токен,
метки или будущую историю через функцию. Новые LLM-признаки не заимствуются из ручных/GNN-признаков.
